# 05b — Uncertainty Calibration · Phase 5B

**Purpose:** Load Phase 5A predictions, apply temperature scaling, evaluate before/after calibration.

**Run AFTER:**
1. `python scripts/phase5a_save_predictions.py` — creates correct NPZ files
2. `python scripts/phase5b_calibrate.py --all` — runs calibration

**Research Gap addressed:** Gap 2 — Calibration

**Key question:** Are the GAT uncertainty intervals well-calibrated?
- 90% prediction interval should contain true value 90% of the time (PICP=0.90)
- Temperature scaling corrects over/underconfidence post-hoc
- No model retraining — only T is learned on validation set
---

# Setup

In [ ]:
import os, sys, pickle
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils.config import load_yaml_config
from wildfire_gnn.models.calibration import (
    TemperatureScaling,
    compute_picp, compute_mpiw,
    reliability_curve, compute_ace, compute_ence,
    compute_all_calibration_metrics,
)

config   = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
p        = config['paths']
PRED_DIR = PROJECT_ROOT / 'reports' / 'predictions'
TBL_DIR  = PROJECT_ROOT / 'reports' / 'tables'
FIG_DIR  = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

ARCH = 'GAT'   # Change to 'GCN' or 'GraphSAGE' for ablation
print(f'Architecture : {ARCH}')
print(f'Predictions  : {PRED_DIR}')